<a href="https://colab.research.google.com/github/archel4759/praktika/blob/main/web_wetlands.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip  install patchify
!pip install rasterio
!pip install imageio
!pip install imagecodecs
!pip install --upgrade tifffile
!pip install Flask
!pip install py7zr
!rm -rf MyDrive
!pip install opencv-python
#!pip install tensorflow
!pip install fiona
!pip install scipy
!pip install py7zr
!pip install pyodbc
!pip install bcrypt

In [ ]:
import os
os.environ["SM_FRAMEWORK"] = "tf.keras"
import cv2
import sys
import patchify
import tifffile
import rasterio
from rasterio.features import shapes as rasterio_shapes
import fiona
import imageio
import glob
from matplotlib import pyplot as plt
from patchify import patchify as pch, unpatchify as unpch
from PIL import Image

In [ ]:
!pip uninstall -y numpy
!pip uninstall -y tensorflow
!pip install numpy==1.26.4
!pip install tensorflow==2.14.0

Перезапуск сессии (ctrl+m)

In [ ]:
import numpy as np
print(np.__version__)
import tensorflow as tf
print(tf.__version__)
import keras as kr
print(kr.__version__)
model = kr.models.load_model('/content/drive/MyDrive/WD_6classes_25epochs_RESNET152_batch4.hdf5', compile=False)
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

1.26.4
2.14.0
2.14.0


In [ ]:
import os

os.makedirs('templates', exist_ok=True)
with open('templates/login.html', 'w') as f:
    f.write('''
    <!DOCTYPE html>
    <html lang="ru">
    <head>
        <meta charset="UTF-8">
        <title>Вход</title>
        <style>
            body {
                font-family: Calibri, sans-serif;
                display: flex;
                justify-content: center;
                align-items: flex-start;
                height: 100vh;
                margin: 0;
            }
            .container {
                margin-top: 50px;
                text-align: center;
            }
        </style>
    </head>
    <body>
        <div class="container">
            <h2>Вход</h2>
            <form method="POST" action="/login">
                <label for="username">Имя пользователя:</label>
                <input type="text" id="username" name="username"><br><br>
                <label for="password">Пароль:</label>
                <input type="password" id="password" name="password"><br><br>
                <input type="submit" value="Войти">
            </form>
            {% if error %}
                <p style="color:red;">{{ error }}</p>
            {% endif %}
             <p>Идентификатор сессии: {{ session_id }}</p>
        </div>
    </body>
    </html>
    ''')

# Сохраните шаблон upload.html
with open('templates/upload.html', 'w') as f:
    f.write('''
<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <title>Загрузка файла</title>
    <style>
        body {
            font-family: Calibri, sans-serif; /* Установка шрифта Calibri */
            display: flex;
            justify-content: center;
            align-items: flex-start;
            height: 100vh;
            margin: 0;
        }
        .container {
            margin-top: 50px;
            text-align: center;
        }
        .hidden {
            display: none;
        }
        button {
            margin-top: 10px;
            padding: 10px 20px;
            font-size: 16px;
            cursor: pointer;
        }
        select {
            margin-top: 10px;
            padding: 10px;
            font-size: 16px;
        }
        .form-section {
            margin-bottom: 20px; /* Интервал между объектами формы */
        }
    </style>
    <script>
        function showOptions() {
            document.getElementById('options').classList.remove('hidden');
            document.getElementById('processButton').classList.add('hidden'); // Скрыть кнопку "Обработать"
            document.getElementById('clientOptions').classList.add('hidden');
            document.getElementById('serverOptions').classList.add('hidden');
        }

        function showClientOptions() {
            document.getElementById('clientOptions').classList.remove('hidden');
            document.getElementById('serverOptions').classList.add('hidden');
            document.getElementById('uploadFromClient').classList.add('hidden');
            document.getElementById('uploadFromServer').classList.remove('hidden');
        }

        function showServerOptions() {
            document.getElementById('serverOptions').classList.remove('hidden');
            document.getElementById('clientOptions').classList.add('hidden');
            document.getElementById('uploadFromServer').classList.add('hidden');
            document.getElementById('uploadFromClient').classList.remove('hidden');
            loadServerFiles();
        }

        function loadServerFiles() {
            fetch('/list_files')
                .then(response => response.json())
                .then(data => {
                    let fileSelect = document.getElementById('fileSelect');
                    fileSelect.innerHTML = '';
                    data.files.forEach(file => {
                        let option = document.createElement('option');
                        option.value = file;
                        option.textContent = file;
                        fileSelect.appendChild(option);
                    });
                })
                .catch(error => {
                    console.error('Ошибка при загрузке файлов с сервера:', error);
                });
        }

        function processClientFile() {
            let fileInput = document.querySelector('input[name="file"]');
            if (!fileInput.value) {
                alert('Пожалуйста, выберите файл для загрузки.');
                return false;
            }
            document.getElementById('uploadForm').submit(); // Подать форму загрузки файла с клиента
        }

        function validateForm(event) {
        let username = document.getElementById('username').value.trim();
        let password = document.getElementById('password').value.trim();
        let role = document.getElementById('role').value.trim();



        function processServerFile() {
            document.getElementById('fileSelectForm').submit(); // Подать форму загрузки файла с сервера
        }
    </script>
</head>
<body>
    <div class="container">
        <h2>Добро пожаловать, {{ user }}</h2>
        <p>Идентификатор сессии: {{ session_id }}</p>
        <p>Флаг обработки: {{ processed }}</p> <!-- Диагностика состояния флага -->

        <!-- Форма для обработки файла -->
        <form method="POST" action="/process" id="processForm">
            <button type="button" id="processButton" onclick="showOptions()">Обработать</button>
        </form>

        <!-- Опции выбора способа загрузки файла для обработки -->
        <div id="options" class="hidden">
            <h3>Выберите способ загрузки файла для обработки:</h3>
            <!-- Большие кнопки -->
            <button id="uploadFromClient" class="form-section" onclick="showClientOptions()">Загрузить с клиента</button>
            {% if role != 'ext' %}
            <button id="uploadFromServer" class="form-section" onclick="showServerOptions()">Загрузить с сервера</button>
            {% endif %}
        </div>

        <!-- Опции для загрузки с клиента -->
        <div id="clientOptions" class="hidden">
            <form method="POST" action="/upload" enctype="multipart/form-data" id="uploadForm">
                <input type="file" name="file" accept=".tif" required><br><br>
                <button type="button" onclick="processClientFile()">Начать обработку клиентского файла</button>
            </form>
        </div>

        <!-- Опции для загрузки с сервера -->
        {% if role != 'ext' %}
        <div id="serverOptions" class="hidden">
            <form method="POST" action="/download_from_drive" id="fileSelectForm">
                <label for="fileSelect">Выберите файл с сервера:</label>
                <select id="fileSelect" name="file">
                    <!-- Опции будут добавлены через JavaScript -->
                </select><br><br>
                <button type="button" onclick="processServerFile()">Начать обработку серверного файла</button>
            </form>
        </div>
        {% endif %}

        {% if processed %}
        <!-- Кнопка для сохранения файла на клиенте -->
        <form method="POST" action="/save_local">
            <button type="submit">Сохранить на клиент</button>
        </form>

        <!-- Кнопка для сохранения файла на сервере -->
        {% if role != 'ext' %}
        <form method="POST" action="/save_drive">
            <button type="submit">Сохранить на сервер</button>
        </form>
        {% endif %}
        {% endif %}

        {% if role == 'adm' %}
        <!-- Кнопка для скачивания лога-->
        <form method="post" action="{{ url_for('download_log') }}">
            <button type="submit">Выгрузить лог</button>
        </form>

        <form method="get" action="{{ url_for('register') }}">
            <button type="submit">Зарегистрировать пользователя</button>
        </form>

        {% endif %}

        {% if error %}
            <p style="color:red;">{{ error }}</p>
        {% endif %}
        {% if message %}
            <p style="color:green;">{{ message }}</p>
        {% endif %}
    </div>
</body>
</html>

    ''')

with open('templates/register.html', 'w') as f:
    f.write('''
<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <title>Регистрация пользователя</title>
    <style>
        body {
            font-family: Calibri, sans-serif;
            display: flex;
            justify-content: center;
            align-items: flex-start;
            height: 100vh;
            margin: 0;
        }
        .container {
            margin-top: 50px;
            text-align: center;
        }
        form {
            margin-top: 20px;
        }
        input, select {
            margin-top: 10px;
            padding: 10px;
            font-size: 16px;
            width: 100%;
            max-width: 300px;
        }
        button {
            margin-top: 20px;
            padding: 10px 20px;
            font-size: 16px;
            cursor: pointer;
        }
        .flash {
            color: red;
            margin-bottom: 20px;
        }
    </style>
</head>
<body>
    <div class="container">
        <h2>Регистрация нового пользователя</h2>

        {% with messages = get_flashed_messages() %}
          {% if messages %}
            <div class="flash">
              {% for message in messages %}
                <p>{{ message }}</p>
              {% endfor %}
            </div>
          {% endif %}
        {% endwith %}

        <form method="POST">
            <label for="username">Имя пользователя:</label><br>
            <input type="text" id="username" name="username" required><br><br>

            <label for="password">Пароль:</label><br>
            <input type="password" id="password" name="password" required><br><br>

            <label for="role">Роль:</label><br>
            <select id="role" name="role" required>
                <option value="adm">adm</option>
                <option value="emp">emp</option>
                <option value="ext">ext</option>
            </select><br><br>

            <button type="submit">Зарегистрировать</button>
        </form>

        <br>
        <a href="{{ url_for('login') }}">На страницу авторизации</a>
    </div>
</body>
</html>
    ''')


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
!pip install pyngrok
!ngrok config add-authtoken 2jkKaaRUcIKjLeWuoqKhtw5w513_zNJavfxTQEBZdyP2V75H

Mounted at /content/drive
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
import sqlite3
from flask import Flask, render_template, request, redirect, url_for, flash, session, send_file, jsonify
from pyngrok import ngrok
import pyodbc
import bcrypt
from werkzeug.utils import secure_filename
from werkzeug.security import generate_password_hash
from werkzeug.utils import secure_filename
from werkzeug.security import check_password_hash
import os
import cv2
import matplotlib.pyplot as plt
import logging
import zipfile
import py7zr
from datetime import datetime, timedelta
import secrets

log_file_path = '/content/drive/MyDrive/folder/session_log.txt'
logging.basicConfig(filename=log_file_path, level=logging.INFO, format='%(asctime)s - %(username)s - %(message)s')

app = Flask(__name__)
app.secret_key = 'sk'
app.config['UPLOAD_FOLDER'] = '/content/drive/MyDrive/upload folder'
app.config['PROCESSED_FOLDER'] = 'processed'
app.config['DRIVE_FOLDER'] = '/content/drive/MyDrive/folder'

def generate_session_id():
    return f"{secrets.token_hex(3).upper()[:3]}-{secrets.token_hex(3).upper()[:3]}"

SESSION_ID = generate_session_id()

def log_action(content):
    with open('/content/drive/MyDrive/folder/session_log.txt', 'a') as f:
        current_time = datetime.now()
        new_time = current_time + timedelta(hours=3)
        timestamp = new_time.strftime('%Y-%m-%d %H:%M:%S')
        f.write(f"{timestamp}/{SESSION_ID}: {content}\n")
        print(f'+ log / {content}')

@app.route('/', methods=['GET', 'POST'])
@app.route('/login', methods=['GET', 'POST'])
def login():
    session['processed'] = False
    if request.method == 'POST':
        username = request.form.get('username', '').strip().lower()
        password = request.form.get('password', '').strip()

        if not username or not password:
            flash('Имя пользователя и пароль должны быть заполнены')
            log_action(f"{username} осуществлена попытка входа с пустым логином и/или паролем")
            return render_template('login.html', error='Имя пользователя и пароль должны быть заполнены', session_id=SESSION_ID)

        try:
            conn = sqlite3.connect('/content/drive/MyDrive/folder/login_db.sqlite')
            cursor = conn.cursor()
            cursor.execute("SELECT password, role FROM login_db WHERE login = ?", (username,))
            result = cursor.fetchone()
            conn.close()

            if result:
                stored_password_hash, role = result
                if check_password_hash(stored_password_hash, password):
                    session['username'] = username
                    session['role'] = role
                    log_action(f"{username} успешно подключился")
                    return redirect(url_for('upload'))
                else:
                    flash('Неверное имя пользователя или пароль')
                    log_action(f"Осущественна безуспешная попытка входа под логином {username} (неверный пароль)")
            else:
                flash('Неверное имя пользователя или пароль')
                log_action(f"Осущественна безуспешная попытка входа под логином {username} (пользователь не найден)")

        except Exception as e:
            flash('Ошибка при попытке входа')
            log_action(f"Ошибка входа для {username}: {e}")

        return render_template('login.html', error='Неверное имя пользователя или пароль', session_id=SESSION_ID)

    return render_template('login.html', session_id=SESSION_ID)

def check_user_exists(username):
    # Проверьте, существует ли уже пользователь с таким именем в базе данных.
    # Здесь должно быть обращение к базе данных, например:
    connection = sqlite3.connect('/content/drive/MyDrive/folder/login_db.sqlite')
    cursor = connection.cursor()
    cursor.execute("SELECT COUNT(1) FROM login_db WHERE login=?", (username,))
    result = cursor.fetchone()
    return result[0] > 0

def add_user_to_db(username, hashed_password, role):
    # Добавьте нового пользователя в базу данных
    connection = sqlite3.connect('/content/drive/MyDrive/folder/login_db.sqlite')
    cursor = connection.cursor()
    cursor.execute("INSERT INTO login_db (login, password, role) VALUES (?, ?, ?)", (username, hashed_password, role))
    connection.commit()
    cursor.close()

@app.route('/register', methods=['GET', 'POST'])
def register():
    if request.method == 'POST':
        username = request.form.get('username').strip().lower()
        password = request.form.get('password').strip()
        role = request.form.get('role')

        if not username or not password or not role:
            flash('Все поля должны быть заполнены')
            return render_template('register.html')

        # Проверка, если пользователь с таким именем уже существует
        if check_user_exists(username):
            flash('Пользователь с таким именем уже существует')
            return render_template('register.html')

        # Хеширование пароля
        hashed_password = generate_password_hash(password)

        # Добавление нового пользователя в базу данных
        add_user_to_db(username, hashed_password, role)

        flash('Пользователь успешно зарегистрирован!')
        return render_template('register.html')

    return render_template('register.html')

@app.route('/upload', methods=['GET', 'POST'])
def upload():
    if 'username' not in session:
        return redirect(url_for('login'))

    username = session['username']
    role = session['role']

    if request.method == 'POST':

        if 'file' in request.files:
            file = request.files['file']
            if file.filename == '':
                flash('Файл не выбран')
                log_action(f"{username} - попытка загрузить и обработать файл")
                return render_template('upload.html', role=role, session_id=SESSION_ID, processed=session.get('processed', False), user=username, error='Файл не выбран')

            if file and file.filename.endswith('.tif'):
                filename = secure_filename(file.filename)
                file_path = os.path.join(app.config['UPLOAD_FOLDER'], filename)
                file.save(file_path)
                flash(f"Файл {filename} успешно загружен с клиента")
                log_action(f"{username} загрузил файл на обработку с клиента: {filename}. Путь: {file_path}")
                session['file_path'] = file_path
                log_action(f"{username} начал обработку клиентского файла {filename}. Путь: {file_path}")
                process(file_path)
                log_action(f"{username} успешно завершил обработку {filename}")
                return render_template('upload.html', role=role, user=username, session_id=SESSION_ID, processed=session.get('processed', False), message=f"Клиентский файл {filename} успешно обработан!")
            else:
                flash('Неправильный формат файла. Требуется .tif')
                return render_template('upload.html', role=role, user=username, session_id=SESSION_ID, processed=session.get('processed', False), error='Не удалось обработать клиентский файл')
    return render_template('upload.html', role=role, user=username, session_id=SESSION_ID, processed=session.get('processed', False))

@app.route('/create_zip_archive', methods=['POST'])
def create_zip_archive(input_filename):

    # Указываем путь к папке с исходными файлами
    source_folder ='/content/drive/MyDrive/download folder'
    # Имя архива
    zip_filename = f"/content/drive/MyDrive/download folder/{input_filename}_completed.zip"

    input_folder = '/content/drive/MyDrive/folder'

    files_added = False  # Флаг для проверки добавления файлов
    # Создаем архив
    with zipfile.ZipFile(zip_filename, 'w') as zipf:
    # Перебираем файлы в исходной папке
        for filename in os.listdir(source_folder):
            # Полный путь к файлу
            file_path = os.path.join(source_folder, filename)
            # Проверяем, если имя файла оканчивается на '_completed' и это файл (не папка)
            if os.path.isfile(file_path) and filename.split('_completed')[0] and filename.split('_completed')[-1].startswith('.') and filename.startswith(input_filename):
                # Добавляем файл в архив
                zipf.write(file_path, arcname=filename)
                print(f'Файл {filename} добавлен в архив {zip_filename}')
                files_added = True

        # Проверяем, добавлены ли файлы в архив
    if not files_added:
        print('Нет файлов для добавления в архив. Убедитесь, что файлы существуют и соответствуют критериям.')

    print(f'Создание архива завершено. Архив сохранен как {zip_filename}')

@app.route('/process', methods=['POST'])
def process(input_imgfile):

    username = session['username']

    input_folder = '/content/drive/MyDrive/folder'
    output_folder = '/content/drive/MyDrive/download folder'
    temp_saved_folder = '/content/drive/MyDrive/temp folder'
    os.makedirs(output_folder, exist_ok=True)
    os.makedirs(temp_saved_folder, exist_ok=True)
    os.makedirs(input_folder, exist_ok=True)

    if not os.path.exists(input_imgfile):
        log_action(f"Ошибка: Файл '{input_imgfile}' не существует.")
    else:
        # Пытаемся прочитать изображение
        img = cv2.imread(input_imgfile)

        # Проверяем, было ли изображение успешно прочитано
        if img is None:
            log_action(f"Ошибка: Не удалось прочитать изображение '{input_imgfile}'")
            session['processed'] = False
            return
        else:
            # Преобразуем изображение из BGR в RGB
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Отображаем изображение
            log_action(f"Изображение {input_imgfile}' прочитано")
            from matplotlib.scale import FuncScaleLog
            #for input_imgfile in image_files:
            filename = os.path.splitext(os.path.basename(input_imgfile))[0]
            print("1")
            #img = cv2.imread(input_imgfile)
            #img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            patch_size = 256

            n_classes = 6

            SIZE_X = (img.shape[1]//patch_size)*patch_size #Nearest size divisible by our patch size
            SIZE_Y = (img.shape[0]//patch_size)*patch_size #Nearest size divisible by our patch size

            large_img = Image.fromarray(img)

            large_img = large_img.crop((0 ,0, SIZE_X, SIZE_Y))  #Crop from top left corner
            large_img = np.array(large_img)

            log_action(f'Началась обработка {filename}')
            patches_img = pch(large_img, (patch_size, patch_size, 3), step=patch_size)  #Step=256 for 256 patches means no overlap
            patches_img = patches_img[:,:,0,:,:,:]
            print("2")
            patched_prediction = []
            print("4")
            for i in range(patches_img.shape[0]):
                for j in range(patches_img.shape[1]):
                    print("5")
                    single_patch_img = patches_img[i,j,:,:,:]
                    print("6")
                    single_patch_img = scaler.fit_transform(single_patch_img.reshape(-1, single_patch_img.shape[-1])).reshape(single_patch_img.shape)
                    single_patch_img = np.expand_dims(single_patch_img, axis=0)
                    print("7")
                    pred = model.predict(single_patch_img)
                    print("7")
                    pred = np.argmax(pred, axis=3)
                    print("8")
                    pred = pred[0, :,:]
                    patched_prediction.append(pred)
            print("3")
            patched_prediction = np.array(patched_prediction)
            patched_prediction = np.reshape(patched_prediction, [patches_img.shape[0], patches_img.shape[1],
                                                        patches_img.shape[2], patches_img.shape[3]])
            unpatched_prediction = unpch(patched_prediction, (large_img.shape[0], large_img.shape[1]))
            temp_saved_prediction = f'{temp_saved_folder}/predicted_{filename}.tif'
            imageio.imsave(temp_saved_prediction, unpatched_prediction)
            plt.imshow(unpatched_prediction, cmap = 'gray')
            plt.axis('off')
            output_image_path = f'{output_folder}/{filename}_completed.tif'
            log_action(f'Обработка нейросетевой моделью завершена')
            print('OUTPUT IMAGE PATH .....')
            print(output_image_path)
            ## Откройте исходное изображение с использованием rasterio
            with rasterio.open(input_imgfile) as src:
                # Получите метаданные и геопривязку исходного изображения
                meta = src.meta
                geotransform = src.transform
                # Откройте кропнутое изображение с использованием rasterio
                with rasterio.open(temp_saved_prediction) as src_cropped:
                    # Прочитайте все бэнды кропнутого изображения
                    cropped_image = src_cropped.read()
                # Обновите метаданные исходного изображения с метаданными кропнутого изображения
                meta.update(width=src_cropped.width, height=src_cropped.height, count=cropped_image.shape[0])
                # Сохраните результирующий датасет с геопривязкой исходного изображения
                with rasterio.open(output_image_path, 'w', **meta) as dst:
                    # Запишите все бэнды кропнутого изображения в результирующий датасет
                    for i in range(cropped_image.shape[0]):
                        dst.write(cropped_image[i, :, :], i + 1)
                    # Установите геопривязку для результирующего датасета
                    dst.transform = geotransform
                    print(f"Геопривязка для {filename} установлена")
                    log_action(f'Геопривязка для {filename} восстановлена')
            # Вывод сообщения об успешном выполнении
            print(f"Полностью завершилась обработка {filename}")
            os.remove(temp_saved_prediction)
            log_action(f'Обработанный растр {filename} сохранен в формате tif')
            print(filename,'сохранен в формате .tif')
            # Формируем путь для сохранения векторного файла
            output_shapefile = os.path.join(output_folder, f'{filename}_completed.shp')
            # Открываем растровое изображение
            with rasterio.open(output_image_path) as src:
                # Получаем геопривязку и количество классов
                transform = src.transform
                num_classes = len(src.colorinterp)
                # Читаем растровые данные и векторизуем
                image = src.read(1)
                vector_shapes = list(rasterio_shapes(image, mask=None, transform=transform))
                # Создаем новый векторный файл
                schema = {
                    'geometry': 'Polygon',
                    'properties': {'class': 'int'},
                }
                with fiona.open(output_shapefile, 'w', 'ESRI Shapefile', schema, crs=src.crs) as output:
                  # Записываем геометрию и атрибуты векторного файла
                  for shape, value in vector_shapes:
                      feature = {
                          'geometry': shape,
                          'properties': {'class': value},
                      }
                      output.write(feature)
            # Выводим информацию о геопривязке и количестве классов
            log_action(f'Векторизация растра {filename} проведена успешно. Shape-файл сохранен')
            print(f"Обработан файл: {filename}. Shape-файл сохранен.")
            print("======================================")


    if 'username' not in session:
        return redirect(url_for('login'))

    file_path = session.get('file_path')
    if not file_path:
        flash('Сначала загрузите файл')
        return render_template('upload.html',role=session['role'], user=session['username'], session_id=SESSION_ID, processed=session.get('processed', False), error='Сначала загрузите файл')

    print(f"DEBUG: Processing file path from session: {file_path}")  # Debug print
    session['processed'] = True
    print(f"session['processed'] : {session['processed']}")  # Debug print

    return

def create_7z_from_files(starting_name, source_folder, archive_filename):
    """
    Создает архив .7z, содержащий файлы с именем, начинающимся на `starting_name` и заканчивающимся на `_completed`,
    из папки `source_folder`.

    :param starting_name: Начальное имя файла для поиска
    :param source_folder: Папка, в которой ищутся файлы
    :param archive_filename: Полное имя создаваемого архива
    """
    log_action(f'Начато создание архива для экспорта')
    valid_extensions = {'.tif', '.shp', '.cpg', '.dbf', '.prj', '.shx'}
    # Создаем архив
    with py7zr.SevenZipFile(archive_filename, 'w') as archive:
        # Перебираем файлы в исходной папке
        files_added = False  # Флаг для проверки добавления файлов
        for filename in os.listdir(source_folder):
            # Полный путь к файлу
            file_path = os.path.join(source_folder, filename)

            # Проверяем, если имя файла оканчивается на '_completed' перед расширением и это файл (не папка)
            if os.path.isfile(file_path):
                base_name, ext = os.path.splitext(filename)
                print(f"base_name -- {base_name}")

                if base_name.endswith('_completed') and base_name.startswith(starting_name):

                    file_extension = os.path.splitext(filename)[1].lower()

                    if file_extension in valid_extensions:
                      archive.write(file_path, arcname=filename)
                      print(f'Файл {filename} добавлен в архив {archive_filename}')
                      log_action(f'Файл {filename} добавлен в архив {archive_filename}')
                      files_added = True

        if not files_added:
            print('Нет файлов для добавления в архив. Убедитесь, что файлы существуют и соответствуют критериям.')
            log_action('Сформирован пустой архив')

    print(f'Создание архива завершено. Архив сохранен как {archive_filename}')
    log_action(f'Создание архива завершено. Архив сохранен как {archive_filename}')

@app.route('/save_local', methods=['POST'])
def save_local():
    if 'username' not in session:
        return redirect(url_for('login'))

    file_path = session.get('file_path')
    if not file_path:
        flash('Сначала загрузите файл')
        return render_template('upload.html',role=session['role'], session_id=SESSION_ID, processed=session.get('processed', False), user=session['username'], error='Сначала загрузите файл')

    # Определяем имя и путь архива


    print(f"Началось создание архива")
    # Создаем ZIP-архив
    # Пример использования функции
    starting_name = os.path.splitext(os.path.basename(file_path) )[0]
    print(f"    starting_name: {starting_name}")
    source_folder = '/content/drive/MyDrive/download folder'
    archive_filename = f'/content/drive/MyDrive/download folder/{starting_name}_completed.7z'
    print(f"    archive_filename: {archive_filename}")
    create_7z_from_files(starting_name,source_folder,archive_filename)

    # Проверяем существование созданного архива
    if not os.path.exists(archive_filename):
        flash('Не удалось создать архив')
        log_action(f"Не удалось создать архив: {archive_filename}")
        print(f"Не удалось создать архив: {archive_filename}")  # Debug print
        return render_template('upload.html',role=session['role'], session_id=SESSION_ID, processed=session.get('processed', False), user=session['username'], error='Не удалось создать архив')

    # Отправляем архив пользователю для скачивания
    log_action('Архив сохранен локально')
    return send_file(archive_filename, as_attachment=True)

@app.route('/save_drive', methods=['POST'])
def save_drive():
    if 'username' not in session:
        return redirect(url_for('login'))

    file_path = session.get('file_path')
    if not file_path:
        flash('Сначала загрузите файл')
        return render_template('upload.html',role=session['role'], session_id=SESSION_ID, processed=session.get('processed', False), user=session['username'], error='Сначала загрузите файл')

    print(f"DEBUG: Saving to drive file path from session: {file_path}")  # Debug print
    print(f"Началось создание архива")
    # Создаем ZIP-архив
    # Пример использования функции
    starting_name = os.path.splitext(os.path.basename(file_path))[0]
    print(f"    starting_name: {starting_name}")
    source_folder = '/content/drive/MyDrive/download folder'
    archive_filename = f'/content/drive/MyDrive/download folder/{starting_name}_completed.7z'
    print(f"    archive_filename: {archive_filename}")
    create_7z_from_files(starting_name,source_folder,archive_filename)

    # Проверяем существование созданного архива
    if not os.path.exists(archive_filename):
        flash('Не удалось создать архив')
        print(f"Не удалось создать архив: {archive_filename}")  # Debug print
        return render_template('upload.html',role=session['role'],session_id=SESSION_ID, processed=session.get('processed', False), user=session['username'], error='Не удалось создать архив')


    try:
        dst_folder = '/content/drive/MyDrive/folder for save'
        dst_path = os.path.join(dst_folder, os.path.basename(archive_filename))
        os.rename(archive_filename, dst_path)
        log_action('Файл успешно сохранен на Google Drive')
        print('Файл успешно сохранен на Google Drive')
    except Exception as e:
        flash(f'Ошибка при сохранении файла на Google Drive: {str(e)}')
        log_action(f'Ошибка при сохранении файла на Google Drive: {str(e)}')
        return render_template('upload.html', session_id=SESSION_ID, processed=session.get('processed', False), user=session['username'], error=f'Ошибка при сохранении файла на Google Drive (archive_filename - {archive_filename}):: {str(e)}')

    log_action('Архив сохранен на сервере (/content/drive/MyDrive/folder for save)')
    return render_template('upload.html',role=session['role'], user=session['username'], session_id=SESSION_ID, processed=session.get('processed', False), message='Файл успешно сохранен на Google Drive (../folder for save)')

@app.route('/download_from_drive', methods=['POST'])
def download_from_drive():
    if 'username' not in session:
        return redirect(url_for('login'))

    file_name = request.form.get('file')
    if not file_name:
        flash('Файл не выбран')
        return render_template('upload.html',role=session['role'], session_id=SESSION_ID, processed=session.get('processed', False), user=session['username'], error='Файл не выбран')

    file_path = os.path.join(app.config['DRIVE_FOLDER'], file_name)
    if not os.path.exists(file_path):
        flash('Файл не найден на сервере')
        return render_template('upload.html',role=session['role'], session_id=SESSION_ID, processed=session.get('processed', False), user=session['username'], error='Файл не найден на сервере')

    # Сохраняем путь к файлу для последующей обработки
    session['file_path'] = file_path
    log_action(f"Начата обработка серверного файла {file_path}")
    process(file_path)
    if session['processed']:
      log_action(f"Успешно завершена обработка серверного файла {file_path}")
      return render_template('upload.html',role=session['role'], user=session['username'], session_id=SESSION_ID, processed=session.get('processed', False), message=f'Серверный файл {file_name} успешно обработан!')
    elif not session['processed']:
      return render_template('upload.html',role=session['role'], user=session['username'], session_id=SESSION_ID, processed=session.get('processed', False), error=f'Не удалось прочитать изображение {file_name}')

@app.route('/download_log', methods=['POST'])
def download_log():
    if session['role'] != 'adm':
        log_action('У вас нет доступа к этому разделу')
        flash('У вас нет доступа к этому разделу')
        return redirect(url_for('upload'))

    log_file_path = '/content/drive/MyDrive/folder/session_log.txt'
    if not os.path.exists(log_file_path):
        log_action('Лог-файл не найден')
        flash('Лог-файл не найден')
        return redirect(url_for('upload'))


    print('SEND_FILE 165')
    log_action(f"{session['username']} выгрузил лог")
    return send_file(log_file_path, as_attachment=True)

@app.route('/list_files', methods=['GET'])
def list_files():
    if 'username' not in session:
        return redirect(url_for('login'))

    try:
        files = os.listdir(app.config['DRIVE_FOLDER'])
        tif_files = [file for file in files if file.endswith('.tif')]
        return jsonify({'files': tif_files})
    except Exception as e:
        return jsonify({'error': str(e)}), 500
# Запустите туннель на 5000 порт
public_url = ngrok.connect(5000)
log_action(f"Сессия запустилась по адресу: {public_url}")
print("URL для доступа к приложению:", public_url)

if __name__ == '__main__':
    app.run(port=5000)


In [ ]:
import sqlite3
conn = sqlite3.connect('/content/drive/MyDrive/folder/login_db.sqlite')
cursor = conn.cursor()

cursor.execute("SELECT * FROM login_db")
rows = cursor.fetchall()

for row in rows:
    print(row)

conn.close()